# 13 - Imbalanced Classification

In this notebook, we explore techniques to force a model to pay attention to rare events (like Fraud) instead of ignoring them to achieve high accuracy.

## Concept Overview
If a dataset is 99% Class 0 and 1% Class 1, a standard model will usually predict 0 every time. We fix this at the **Algorithm Level** (Class Weights) or the **Data Level** (SMOTE).

## Visualizing the Problem
Let's train a standard Random Forest on imbalanced data.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Generate 1% minority class data
X, y = make_classification(n_samples=10000, weights=[0.99, 0.01], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train standard model
model_standard = RandomForestClassifier(random_state=42)
model_standard.fit(X_train, y_train)
y_pred = model_standard.predict(X_test)

print("Standard Model Report:")
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
plt.title('Standard Model (Fails on Class 1)')
plt.show()

Notice how the model completely failed to find any Class 1 instances (Recall is 0.0). It ignored the minority class entirely.

## Solution 1: Algorithmic Level (Class Weights)
We tell the algorithm that making a mistake on Class 1 is 100 times worse than making a mistake on Class 0.

In [ ]:
# Using class_weight='balanced'
model_weighted = RandomForestClassifier(class_weight='balanced', random_state=42)
model_weighted.fit(X_train, y_train)
y_pred_w = model_weighted.predict(X_test)

print("Weighted Model Report:")
print(classification_report(y_test, y_pred_w))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_w, cmap='Reds')
plt.title('Weighted Model')
plt.show()

The Recall for Class 1 improved significantly! However, the Precision dropped. This is the tradeoff of forcing the model to be more sensitive.

## Solution 2: Data Level (SMOTE)
If weights aren't enough, we synthesize new fake data points using SMOTE. 

*CRITICAL: ONLY APPLY SMOTE TO THE TRAINING SET!*

In [ ]:
from imblearn.over_sampling import SMOTE

# 1. Apply SMOTE ONLY to X_train
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original Train Set Class 1 Count: {sum(y_train)}")
print(f"SMOTE Train Set Class 1 Count:    {sum(y_train_smote)}")

# 2. Train on the SMOTE data
model_smote = RandomForestClassifier(random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

# 3. Evaluate on the untouched, highly imbalanced X_test
y_pred_s = model_smote.predict(X_test)

print("\nSMOTE Model Report:")
print(classification_report(y_test, y_pred_s))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_s, cmap='Greens')
plt.title('SMOTE Model')
plt.show()

## Industry Notes
Many Junior Data Scientists apply SMOTE to the entire `X` dataset *before* `train_test_split`. This causes synthetic data to leak into the Test set, resulting in artificially high scores that will crash in production. Always split first.

## Exercises
1. Try `RandomUnderSampler` from `imblearn.under_sampling`. Compare its speed and performance to SMOTE.
2. Evaluate all three models above using the `balanced_accuracy_score` metric instead of the classification report.